# 01 · Data preparation and leakage audit
Loads every configured dataset, harmonises labels to one taxonomy, removes duplicates **before** splitting, and checks the splits.

Outputs per dataset (in `results/<dataset>/`): `load_info.json`, `clean_raw.json`, `clean_dedup.json`, `profile.csv`, `split_summary.csv`, `features.txt`.

**Label taxonomy** (shared by all datasets): Normal · DDoS · DoS · Probe (PortScan) · BFA (FTP/SSH brute force, Patator) · Web-Attack · BOTNET · U2R · Other (Infiltration, Heartbleed).

In [ ]:
import sys, os, json, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
pd.set_option('display.width', 200, 'display.max_columns', 60)
from sdnids_bench.config import load_config, mode_params, active_datasets, enable_determinism
cfg = load_config('..')
mp = mode_params(cfg)
enable_determinism()
print('mode:', cfg['mode'], '| results:', cfg['out_dir'])
from sdnids_bench import datasets as D
from sdnids_bench.experiments import load_clean, ds_dir, prepare
DATASETS = active_datasets(cfg)
print('datasets:', DATASETS)

## 1. Load and clean (cached after the first run)

In [ ]:
profiles = {}
for ds in DATASETS:
    raw, classes = load_clean(cfg, ds, dedup=False)
    ded, _ = load_clean(cfg, ds, dedup=True)
    base = ds_dir(cfg, ds)
    li = json.load(open(f'{base}/load_info.json'))
    cr, cd = json.load(open(f'{base}/clean_raw.json')), json.load(open(f'{base}/clean_dedup.json'))
    prof = D.dataset_profile(li, cr, cd, classes)
    prof.to_csv(f'{base}/profile.csv', index=False)
    profiles[ds] = prof
    print(f'\n=== {ds} ===  files: {len(li["files"])}, common features: {li["n_features_common"]}')
    print('timestamp parsing:', {k: v['format'] for k, v in li['timestamp_parsing'].items()})
    print(f'exact duplicates removed: {cd["exact_duplicates_removed"]:,}; conflicting rows removed: {cd["conflicting_rows_removed"]:,}; non-finite values replaced: {cr["nonfinite_values_replaced"]:,}')
    if li['dropped_noncommon_features']:
        print('features not common to all files (dropped):', li['dropped_noncommon_features'])
    display(prof)

## 2. Split audit (S-C at the tuning window length)
Train and validation counts are after the configured `train_cap` / `val_cap` (test is never capped). Checks: every class appears in train, validation and test; how many test flows the rare classes have; which classes were too small for the embargo; and **context-key purity** — the share of test flows whose destination host only ever receives one class. High purity (> ~95%) means windows could leak the label through the host; if so, set `window_key` to `"pair"` in `config.json` and re-run.

In [ ]:
from collections import Counter
summ = []
for ds in DATASETS:
    d = prepare(cfg, ds, 'SC', seed=0, L=cfg['tune_L'], task='multi')
    for part in ('tr', 'va', 'te'):
        cnt = Counter(d[part]['y'].tolist())
        for i, c in enumerate(d['classes']):
            summ.append({'dataset': ds, 'partition': {'tr':'train','va':'validation','te':'test'}[part], 'class': c, 'flows': cnt.get(i, 0)})
    print(f"{ds}: features after variance filter = {len(d['features'])}; test-host purity = {d['purity_te']:.3f}; classes without embargo: {d['no_embargo_classes']}")
    with open(os.path.join(ds_dir(cfg, ds), 'features.txt'), 'w') as f:
        f.write('\n'.join(d['features']))
    del d
summ = pd.DataFrame(summ)
tab = summ.pivot_table(index=['dataset','class'], columns='partition', values='flows', aggfunc='sum')[['train','validation','test']]
for ds in DATASETS:
    tab.loc[ds].to_csv(os.path.join(ds_dir(cfg, ds), 'split_summary.csv'))
tab

Rare classes with fewer than ~10 test flows are reported descriptively only (exact counts with Wilson intervals), as declared in the design.